# cAPTure: versioned graph OOF logit rescoring

This notebook repeats outer-fold inference from the eight immutable final checkpoints for Edge MLP, EdgeGRU, StaticGNN, and ST-GNN. It preserves the original float32 probability, the recomputed float32 probability, the raw model logit, and a sigmoid computed in float64.

No model is trained or refitted, no optimizer is created, and no threshold is selected. The original training OOF files, comparison, and saturation audit remain unchanged. Test1, Test2, `train_pub_exf`, and `train_user_prop` remain prohibited.

## 1. Mount Drive and prepare a GPU runtime

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_rescoring_inputs")
LOCAL_WORK_ROOT = Path("/content/capture_graph_rescoring_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
import torch
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_oof_rescoring import (
    expected_rescoring_job_ids,
    finalize_capture_graph_oof_rescoring,
    load_graph_oof_rescoring_config,
    run_capture_graph_oof_rescoring_job,
    validate_capture_graph_oof_rescoring,
)
from utils.capture_graph_score_saturation import (
    validate_graph_score_saturation_audit,
)

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before rescoring.")
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CUDA device:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Repository commit: 61864da8af1656703684f2ff275b62103240b352
CUDA device: Tesla T4


## 2. Bind the immutable source artifacts

In [2]:
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
TRAINING_BINDING_RUN_ID = (
    "20260929T144234_934044Z_graph_training_binding"
)
COMPARISON_RUN_ID = (
    "20261001T153351_608797Z_graph_four_model_comparison"
)
SATURATION_AUDIT_RUN_ID = (
    "20261001T175306_830261Z_graph_score_saturation_audit"
)
RESCORING_RUN_ID = "20261001_graph_oof_logit_rescoring_v1"

GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
TRAINING_ROOT = (
    DRIVE_ROOT / "graph_training_runs" / TRAINING_BINDING_RUN_ID
)
COMPARISON_DIR = (
    DRIVE_ROOT / "graph_comparison_runs" / COMPARISON_RUN_ID
)
SATURATION_AUDIT_DIR = (
    DRIVE_ROOT / "graph_score_saturation_audit_runs"
    / SATURATION_AUDIT_RUN_ID
)
RESCORING_OUTPUT_ROOT = (
    DRIVE_ROOT / "graph_oof_rescoring_runs" / RESCORING_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
RESCORING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_oof_rescoring_v1.yaml"
)

for path in (
    GRAPH_MATERIALIZATION_DIR, PREPARED_RUN_DIR, TRAINING_ROOT,
    COMPARISON_DIR, SATURATION_AUDIT_DIR, GRAPH_INPUT_CONTRACT_PATH,
    RESCORING_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")

rescoring_config = load_graph_oof_rescoring_config(RESCORING_CONFIG_PATH)
saturation_audit = validate_graph_score_saturation_audit(
    comparison_dir=COMPARISON_DIR,
    output_dir=SATURATION_AUDIT_DIR,
)
print("Saturation-audit status:", saturation_audit["status"])
print("Models requiring score-resolution correction:",
      saturation_audit["saturated_models"])
print("Rescoring output:", RESCORING_OUTPUT_ROOT)

Saturation-audit status: score_saturation_confirmed
Models requiring score-resolution correction: ['edge_gru', 'st_gnn']
Rescoring output: /content/drive/MyDrive/capture_gate0/graph_oof_rescoring_runs/20261001_graph_oof_logit_rescoring_v1


### 2.1 One-time reset of the incomplete rescoring run

Use this cell only to replace the incomplete run created under the earlier numerical-drift bound. It deletes exactly `RESCORING_OUTPUT_ROOT`; it cannot delete the training checkpoints, graph materialization, comparison, or saturation audit. Set both values, execute the cell once, and then return `RESET_EXISTING_RESCORING_RUN` to `False` before any later full-notebook rerun.

In [3]:
# RESET_EXISTING_RESCORING_RUN = True
# RESET_CONFIRMATION = "20261001_graph_oof_logit_rescoring_v1"

RESET_EXISTING_RESCORING_RUN = False
RESET_CONFIRMATION = ""

if RESET_EXISTING_RESCORING_RUN:
    if RESET_CONFIRMATION != RESCORING_RUN_ID:
        raise ValueError(
            "RESET_CONFIRMATION must exactly equal RESCORING_RUN_ID."
        )
    expected_parent = (DRIVE_ROOT / "graph_oof_rescoring_runs").resolve()
    resolved_output = RESCORING_OUTPUT_ROOT.resolve()
    if (
        resolved_output.parent != expected_parent
        or resolved_output.name != RESCORING_RUN_ID
    ):
        raise RuntimeError("Refusing to reset an unexpected output path.")
    if resolved_output.exists():
        shutil.rmtree(resolved_output)
        print("Deleted incomplete rescoring run:", resolved_output)
    else:
        print("No existing rescoring run required deletion.")
else:
    print("Existing rescoring output was not modified.")

Deleted incomplete rescoring run: /content/drive/MyDrive/capture_gate0/graph_oof_rescoring_runs/20261001_graph_oof_logit_rescoring_v1


## 3. Stage and checksum-verify the complete graph collection

Only the complete-fold graph materialization is needed. Selection features are not loaded because no epoch selection or training occurs.

In [4]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
print("Verified graph collection:", collection.root)
print("All graph artifact checksums verified:",
      collection.artifact_checksums_verified)

Staged and verified graph materialization: /content/capture_graph_rescoring_inputs/20260928T175821_521784Z_graph_materialization_full_dev
Verified graph collection: /content/capture_graph_rescoring_inputs/20260928T175821_521784Z_graph_materialization_full_dev
All graph artifact checksums verified: True


## 4. Review and authorize the eight inference-only jobs

The default list contains all four models and both folds. Completed jobs are immutable and are reused on a rerun, so the same cell safely resumes after a runtime interruption. Set `RUN_RESCORING=True` only after checking the paths and job list.

In [5]:
ALL_JOB_IDS = expected_rescoring_job_ids(rescoring_config)
JOB_IDS_TO_RUN = list(ALL_JOB_IDS)
RUN_RESCORING = True

job_table = pd.DataFrame([
    {
        "job_id": job_id,
        "model": job_id.rsplit("__fold_", 1)[0],
        "fold": job_id.rsplit("__fold_", 1)[1],
        "outer_validation_scenarios": rescoring_config["folds"][
            job_id.rsplit("__fold_", 1)[1]
        ]["outer_validation"],
    }
    for job_id in ALL_JOB_IDS
])
display(job_table)
if not JOB_IDS_TO_RUN or not set(JOB_IDS_TO_RUN).issubset(ALL_JOB_IDS):
    raise ValueError("JOB_IDS_TO_RUN contains an unknown or empty selection.")
print("Jobs selected:", JOB_IDS_TO_RUN)
print("Set RUN_RESCORING=True to begin or resume inference.")

,job_id,model,fold,outer_validation_scenarios
0,edge_mlp__fold_A,edge_mlp,A,"[train_dollar_char, train_slash_char, train_su..."
1,edge_mlp__fold_B,edge_mlp,B,"[train_empty_conn, train_qos_mid]"
2,edge_gru__fold_A,edge_gru,A,"[train_dollar_char, train_slash_char, train_su..."
3,edge_gru__fold_B,edge_gru,B,"[train_empty_conn, train_qos_mid]"
4,static_gnn__fold_A,static_gnn,A,"[train_dollar_char, train_slash_char, train_su..."
5,static_gnn__fold_B,static_gnn,B,"[train_empty_conn, train_qos_mid]"
6,st_gnn__fold_A,st_gnn,A,"[train_dollar_char, train_slash_char, train_su..."
7,st_gnn__fold_B,st_gnn,B,"[train_empty_conn, train_qos_mid]"


Jobs selected: ['edge_mlp__fold_A', 'edge_mlp__fold_B', 'edge_gru__fold_A', 'edge_gru__fold_B', 'static_gnn__fold_A', 'static_gnn__fold_B', 'st_gnn__fold_A', 'st_gnn__fold_B']
Set RUN_RESCORING=True to begin or resume inference.


## 5. Run or resume checkpoint rescoring

Each scenario is committed independently to Drive. If the runtime disconnects, rerun sections 1–4 with the same `RESCORING_RUN_ID`, leave the same jobs selected, and execute this cell again. Completed scenarios and jobs will be checksum-verified and reused. The run is sealed automatically only when all eight jobs are complete.

In [6]:
if not RUN_RESCORING:
    print("Rescoring is disabled. No checkpoint was loaded.")
else:
    job_completions = []
    for job_id in JOB_IDS_TO_RUN:
        print(f"\nStarting or resuming {job_id}...")
        completion = run_capture_graph_oof_rescoring_job(
            job_id=job_id,
            config_path=RESCORING_CONFIG_PATH,
            comparison_dir=COMPARISON_DIR,
            saturation_audit_dir=SATURATION_AUDIT_DIR,
            training_root=TRAINING_ROOT,
            prepared_run_dir=PREPARED_RUN_DIR,
            collection=collection,
            output_root=RESCORING_OUTPUT_ROOT,
            local_work_root=LOCAL_WORK_ROOT,
        )
        job_completions.append(completion)
    display(pd.DataFrame([{
        "job_id": item["job_id"],
        "status": item["status"],
        "model_training_performed": item["model_training_performed"],
        "held_out_scenarios_accessed": item["held_out_scenarios_accessed"],
    } for item in job_completions]))


Starting or resuming edge_mlp__fold_A...
Completed logit inference for train_dollar_char: AP=0.895614, source max |delta|=0, 16.0s
Completed logit inference for train_slash_char: AP=0.994230, source max |delta|=0, 21.5s
Completed logit inference for train_sub_exf: AP=0.946288, source max |delta|=0, 15.2s
Completed immutable OOF logit rescoring: edge_mlp__fold_A

Starting or resuming edge_mlp__fold_B...
Completed logit inference for train_empty_conn: AP=0.982297, source max |delta|=0, 13.8s
Completed logit inference for train_qos_mid: AP=0.964709, source max |delta|=0, 14.3s
Completed immutable OOF logit rescoring: edge_mlp__fold_B

Starting or resuming edge_gru__fold_A...
Completed logit inference for train_dollar_char: AP=0.991187, source max |delta|=8.94e-07, 31.7s
Completed logit inference for train_slash_char: AP=0.999969, source max |delta|=3.58e-07, 37.3s
Completed logit inference for train_sub_exf: AP=0.996221, source max |delta|=2.38e-07, 31.0s
Completed immutable OOF logit re

,job_id,status,model_training_performed,held_out_scenarios_accessed
0,edge_mlp__fold_A,complete,False,False
1,edge_mlp__fold_B,complete,False,False
2,edge_gru__fold_A,complete,False,False
3,edge_gru__fold_B,complete,False,False
4,static_gnn__fold_A,complete,False,False
5,static_gnn__fold_B,complete,False,False
6,st_gnn__fold_A,complete,False,False
7,st_gnn__fold_B,complete,False,False


## 6. Seal and review the completed run

This section seals the run only after every expected job exists. If only a subset was requested, it reports the remaining jobs without changing the completed artifacts.

In [7]:
completed_job_ids = [
    job_id for job_id in ALL_JOB_IDS
    if (RESCORING_OUTPUT_ROOT / job_id / "completion.json").is_file()
]
remaining_job_ids = [
    job_id for job_id in ALL_JOB_IDS if job_id not in completed_job_ids
]
print("Completed jobs:", completed_job_ids)
print("Remaining jobs:", remaining_job_ids)

if remaining_job_ids:
    print("The rescoring run remains open until all eight jobs are complete.")
else:
    if (RESCORING_OUTPUT_ROOT / "run_status.json").is_file():
        rescoring_manifest = validate_capture_graph_oof_rescoring(
            config_path=RESCORING_CONFIG_PATH,
            comparison_dir=COMPARISON_DIR,
            saturation_audit_dir=SATURATION_AUDIT_DIR,
            output_root=RESCORING_OUTPUT_ROOT,
        )
    else:
        rescoring_manifest = finalize_capture_graph_oof_rescoring(
            config_path=RESCORING_CONFIG_PATH,
            comparison_dir=COMPARISON_DIR,
            saturation_audit_dir=SATURATION_AUDIT_DIR,
            training_root=TRAINING_ROOT,
            prepared_run_dir=PREPARED_RUN_DIR,
            collection=collection,
            output_root=RESCORING_OUTPUT_ROOT,
        )
    display(pd.DataFrame(rescoring_manifest["jobs"]).T)
    scenario_rows = []
    for job_id in ALL_JOB_IDS:
        job_report = json.loads(
            (RESCORING_OUTPUT_ROOT / job_id / "job_report.json").read_text(
                encoding="utf-8"
            )
        )
        for scenario, metrics in job_report["scenarios"].items():
            scenario_rows.append({
                "model": job_report["model"],
                "fold": job_report["fold"],
                "scenario": scenario,
                "edges": metrics["edges"],
                "raw_logit_ap": metrics["average_precision_raw_logit"],
                "float64_score_ap": metrics["average_precision_score_float64"],
                "source_exact_match_fraction": metrics["source_probability_exact_match_fraction"],
                "source_maximum_absolute_difference": metrics["source_probability_maximum_absolute_difference"],
                "source_exact_one_count": metrics["source_probability_exact_one_count"],
                "float64_exact_one_count": metrics["score_float64_exact_one_count"],
                "raw_logit_minimum": metrics["raw_logit_minimum"],
                "raw_logit_maximum": metrics["raw_logit_maximum"],
            })
    display(pd.DataFrame(scenario_rows))
    print("Rescoring status:", rescoring_manifest["status"])
    print("Operational ranking field:",
          rescoring_manifest["operational_ranking_field"])
    print("Model training performed:",
          rescoring_manifest["model_training_performed"])
    print("Threshold selected:", rescoring_manifest["threshold_selected"])
    print("Held-out scenarios accessed:",
          rescoring_manifest["held_out_scenarios_accessed"])
    print("Next action:", rescoring_manifest["next_action"])

Completed jobs: ['edge_mlp__fold_A', 'edge_mlp__fold_B', 'edge_gru__fold_A', 'edge_gru__fold_B', 'static_gnn__fold_A', 'static_gnn__fold_B', 'st_gnn__fold_A', 'st_gnn__fold_B']
Remaining jobs: []


,model,fold,completion_sha256,job_report_sha256,source_final_checkpoint_sha256,pure_inference_seconds,total_edges
edge_mlp__fold_A,edge_mlp,A,c3af2a59bba5a736fbfa4d0a7830fd41240e6707f4f00d...,c26d861d0bc40dfc5fd5fcf018cd4929a9b1bb673a4fdd...,054d7649dc1885e550f4764b3a2adaa88d268f02be4cf8...,52.701203,11533878
edge_mlp__fold_B,edge_mlp,B,390cc08d97ce8440e061f3605f9e79f1d299a250df4bb0...,a07ea01216b35a7ecae390f1fa9f323b8b647b63af6cf6...,36e0e98e312366c5e8361f9e0644b58314a62370408d47...,28.120099,2675496
edge_gru__fold_A,edge_gru,A,aacbdac8e543fa42608a5eab5de2993965cfe05eafa30d...,5da34c08cf968f4b98422a841fcb51abc36b5832a9adba...,d644c9c9438db3448c2a456ee2400d431860f44f99633a...,100.064325,11533878
edge_gru__fold_B,edge_gru,B,8e0db5d584e0c66caf72856f94e054a1655e00c95eb59c...,06a5f4952f7adc75af30c6db3496ad8c75175eb22db12c...,16e9d4a7d716d1138ef34cb9cce73830854f6a09b67dd8...,61.135998,2675496
static_gnn__fold_A,static_gnn,A,5634e8f0c0d24b015116df821c4737b2def0fa7c030143...,a08e8192bb09ec4639e62d8082338b3394888fa01545f2...,abd935eeb625ed2e6327d1f2abb9b7b0fba8efdb82dc72...,152.265888,11533878
static_gnn__fold_B,static_gnn,B,87b3c3f300ed75bc1e75e403a30f5ec6164e41ea23a6d4...,65f150421eff19f4a46d8c03d21832b53c90c9b62e374a...,cdd8109d1a07bd4cfa95ca199c43a5827c71e36823711b...,94.8452,2675496
st_gnn__fold_A,st_gnn,A,137febc22fd0824d5bd6ca443e1e6c04745e4f448f39a1...,76cd4deeba20b4f235069b7c9d9d4fd8bcda8d04a7f2f8...,e5f9c64bfd8223214f4449cbafd8a84dac841fd03e943d...,186.428088,11533878
st_gnn__fold_B,st_gnn,B,533e378718d28c3fafaf2a27f275e95e1da12a342eeeb5...,93337bc1eafe2115c3fe9c3e90c4142defade0be188482...,fdbbccf23b7bc1233f4f3df6cbb152a879902e3e04918c...,120.882045,2675496


,model,fold,scenario,edges,raw_logit_ap,float64_score_ap,source_exact_match_fraction,source_maximum_absolute_difference,source_exact_one_count,float64_exact_one_count,raw_logit_minimum,raw_logit_maximum
0,edge_mlp,A,train_dollar_char,2882555,0.895614,0.895614,1.000000,0.000000e+00,0,0,-7.783275,6.101065
1,edge_mlp,A,train_slash_char,6425516,0.994230,0.994230,1.000000,0.000000e+00,0,0,-7.783275,6.101064
2,edge_mlp,A,train_sub_exf,2225807,0.946288,0.946288,1.000000,0.000000e+00,0,0,-7.783275,6.101065
3,edge_mlp,B,train_empty_conn,1175779,0.982297,0.982297,1.000000,0.000000e+00,0,0,-16.669395,14.216803
4,edge_mlp,B,train_qos_mid,1499717,0.964709,0.964709,1.000000,0.000000e+00,0,0,-16.669395,14.216803
5,edge_gru,A,train_dollar_char,2882555,0.991187,0.991187,0.838820,8.940697e-07,427776,7,-43.104370,39.638981
6,edge_gru,A,train_slash_char,6425516,0.999969,0.999969,0.977927,3.576279e-07,4696100,0,-43.104370,36.115807
7,edge_gru,A,train_sub_exf,2225807,0.996221,0.996221,0.937001,2.384186e-07,412563,0,-43.104370,34.210255
8,edge_gru,B,train_empty_conn,1175779,0.995354,0.995354,0.891632,2.384186e-07,37178,0,-16.040071,17.774080
9,edge_gru,B,train_qos_mid,1499717,0.994317,0.994317,0.905750,2.682209e-07,47608,0,-16.040071,17.462816


Rescoring status: complete
Operational ranking field: raw_logit
Model training performed: False
Threshold selected: False
Held-out scenarios accessed: False
Next action: run_the_versioned_operational_comparison_on_raw_logits
